In [2]:
import pandas as pd

df_mutants = pd.read_csv('data/mutants.csv')
df = pd.read_csv('data/db_export/submissions.csv')
games_df = pd.read_csv('data/db_export/battleground_games.csv')

In [ ]:
df['game_id'] = df['game_id'].astype(int)
df['objective'] = df['objective'].astype(int)
df['turn'] = df['turn'].astype(int)


games_df['game_id'] = games_df['game_id'].astype(int)
games_df['id'] = games_df['id'].astype(int)
games_df = games_df[games_df['id'] >= 174]

games_df = games_df[~games_df['attacker'].isin(['gpt-4.1-nano'])]
games_df = games_df[~games_df['defender'].isin(['gpt-4.1-nano'])]

games_df = games_df.sort_values(by=['class_alias', 'attacker', 'defender'])

games_df = games_df.loc[~games_df.duplicated(
    subset=['class_alias', 'attacker', 'defender'], keep='last')]

In [ ]:
import numpy as np
import json

invalid_submissions = []
stillborn_submissions = []
mutants = []

for game_id, game in df.groupby('game_id'):
    if not (game_id in games_df['game_id'].values):
        continue
    game_settings = games_df[games_df['game_id'] == game_id].iloc[0]
    game_alias = game_settings['class_alias']
    attacker = game_settings['attacker']
    defender = game_settings['defender']

    game = game[game['side'] == 'attacker']

    for index, row in game.iterrows():
        if row['state'] == 0:
            invalid_submissions.append((game_id, game_alias,
                                        attacker, defender, row['turn'], row['attempt'], row['submission'], row['codedefenders_response']))

        if row['state'] == 1:
            stillborn_submissions.append((game_id, game_alias,
                                         attacker, defender, row['turn'], row['attempt'], row['submission'], row['codedefenders_response']))
        
        if row['state'] == 2:
            mutants.append((game_id, game_alias, attacker, defender,
                            row['turn'], json.loads(row['codedefenders_response'])['mutant']['mutantId'],
                            json.loads(row['codedefenders_response'])['mutant']['diff'],
                            json.loads(row['codedefenders_response'])['mutant']['modifiedLines']))

df_invalid_submissions = pd.DataFrame(invalid_submissions, columns=[
    'game_id', 'class_alias', 'attacker', 'defender', 'turn', 'attempt', 'submission', 'codedefenders_response'])

df_stillborn_submissions = pd.DataFrame(stillborn_submissions, columns=[
    'game_id', 'class_alias', 'attacker', 'defender', 'turn', 'attempt', 'submission', 'codedefenders_response'])

df_diffs = pd.DataFrame(mutants, columns=[
    'game_id', 'class_alias', 'attacker', 'defender', 'turn', 'mutant_id', 'diff', 'modified_lines'])


In [8]:
df_invalid_submissions.to_json('data/invalid_submissions.json', orient='records', lines=True)

In [9]:
df_stillborn_submissions.to_json('data/stillborn_submissions.json', orient='records', lines=True)

In [10]:
df_diffs = df_diffs.merge(df_mutants[['equivalent', 'mutant_id']], on='mutant_id', how='left')

df_diffs.to_json('data/mutants_diffs.json', orient='records', lines=True)